### 0 loading the RE² dataset and first diagnose
because the papers.zip is separate from the main dataset i have to create a column paper text separately


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Loading RE² Dataset
from datasets import load_dataset
import json
import zipfile
from collections import Counter

SEED = 42
ds = load_dataset("Daoze/ReviewRebuttal",
                  data_files="REVIEWS_train.json",
                  split="train",
                  cache_dir='./my_dataset')
print(ds.column_names)
print(ds[0])

# slicing 5000 samples
# sample = ds.select(range(min(5000, len(ds))))
sample = ds.shuffle(seed=SEED).select(range(min(2000, len(ds))))
print(f"{len(sample)} samples loaded")


README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

REVIEWS_train.json: reconstructing file:   0%|          |  0.00B /  319MB            

REVIEWS_train.json: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

['paper_id', 'conference_year_track', 'reviews', 'review_initial_ratings_unified', 'review_final_ratings_unified', 'metareview', 'decision']
{'paper_id': '00I1UhSk55i', 'conference_year_track': 'AAAI 2022 Workshop ADAM', 'reviews': [{'reviewer_id': 'AnonReviewer', 'review_title': 'Well written paper on RL for TO', 'review_content': 'This paper presents a RL approach for solving TO problems in 2D. A sensible framing of the state space, action space is presented. By formulating the transition probability in terms of convolutional operators, the authors are able to smartly utilize a lower resolution FE solver to learn the transition probability. This along with progressive refinement results in a very competitive approach.\n\nA couple of clarifications\n\n- Can the authors show a result with different locations of the 2 bounded elements? While the authors claim that the bounded elements are randomly assigned, every result shown only has bounded elements on the two edges of the domain. In 

In [ ]:
# checking unique ids
ids = {r['paper_id'] for r in sample}
print(f"{len(ids)} unique ids")

2000 unique ids


In [ ]:
# Loading and scanning papers.zip without extracting everything
from huggingface_hub import hf_hub_download
local_zip_path = hf_hub_download(
    repo_id="Daoze/ReviewRebuttal",
    filename="papers.zip",
    repo_type="dataset"
)
print(f"Downloaded to {local_zip_path}")



papers.zip: reconstructing file:   0%|          |  0.00B / 12.4GB            

papers.zip: downloading bytes:           |  0.00B            

Downloaded to /root/.cache/huggingface/hub/datasets--Daoze--ReviewRebuttal/snapshots/0171bacee24a9d405792a32231dabcdb3788b365/papers.zip


In [ ]:
# checking total papers and if any matches the paper id from the original dataset
zf = zipfile.ZipFile(f"{local_zip_path}", "r")
all_paths = zf.namelist()
print(f"papers.zip contains {len(all_paths)} files total")

papers.zip contains 373425 files total


In [ ]:
id_set = set(ids)
id_to_paths = {}

for path in all_paths:
  parts = path.split("/")
  for part in parts[:-1]:
    if part in id_set:
      id_to_paths.setdefault(part, []).append(path)
      break

In [ ]:
print(len(id_to_paths.keys()))

2000


In [ ]:
# Diagnozing match rate
matched_ids = set(id_to_paths.keys())
unmatched_ids = ids - matched_ids
print(f"Matched {len(matched_ids)}/{len(ids)} paper ids "
      f"{len(matched_ids)/len(ids):.1%}")

Matched 2000/2000 paper ids 100.0%


In [ ]:
# counting format availability
fmta_counts = Counter()
for pid, path in id_to_paths.items():
  has_tex = any(p.endswith(".tex") for p in path)
  has_md = any(p.endswith(".md") for p in path)
  fmta_counts[(has_tex, has_md)] += 1
print("has_tex, has_md, counted", dict(fmta_counts))

has_tex, has_md, counted {(True, True): 2000}


In [ ]:
# Extracting matched files
paper_text ={}
extraction_errors=[]
for pid, path in id_to_paths.items():
  tex_paths = [p for p in path if p.endswith(".tex")]
  md_paths =  [p for p in path if p.endswith(".md")]
  chosen = md_paths[0] if md_paths else (tex_paths[0] if tex_paths else None)
  if chosen is None:
    continue
  try:
    with zf.open(chosen) as fh:
      paper_text[pid] = fh.read().decode("utf-8", errors = "replace")
  except Exception as e:
    extraction_errors.append((pid, chosen, str(e)))
print(f"Extracted text for {len(paper_text)} paper_ids")
if extraction_errors:
  print(f"{len(extraction_errors)}Extraction errors for inspection ")
print(extraction_errors[:5])

Extracted text for 2000 paper_ids
[]


In [ ]:
# Sanity Check for Text length
lengths = [len(t) for t in paper_text.values()]
if lengths:
  print(f"\n min char lenth {min(lengths)} \n max char length {max(lengths)} \n avg char length {sum(lengths)/len(lengths)}")
print(f"rought token estimate (4 char = 1 token) : {(sum(lengths)/len(lengths))/4}")



 min char lenth 1737 
 max char length 368843 
 avg char length 59714.3365
rought token estimate (4 char = 1 token) : 14928.584125


In [ ]:
import re

# Broader markers for both references and appendix — covers numbered
# headings, bold-only lines, and common LaTeX bibliography commands
# your first pass missed (printbibliography, bibliographystyle, \appendix).
REF_PATTERN = re.compile(
    r'(\\bibliography\{|\\bibliographystyle\{|\\printbibliography|'
    r'\\begin\{thebibliography\}|'
    r'^#{1,4}\s*(?:[\dIVXLC]+\.?\s*)?references?\s*$|'
    r'^\*{0,2}\s*references?\s*\*{0,2}\s*$)',
    re.IGNORECASE | re.MULTILINE
)


def strip_refs_and_appendix(text):
    ref_match = REF_PATTERN.search(text)
    return text[:ref_match.start()].rstrip() if ref_match else text

before_lens, after_lens = [], []
no_match_ids = []
for pid, text in paper_text.items():
    stripped = strip_refs_and_appendix(text)
    before_lens.append(len(text))
    after_lens.append(len(stripped))
    if len(stripped) == len(text):
        no_match_ids.append(pid)

avg_before = sum(before_lens) / len(before_lens)
avg_after = sum(after_lens) / len(after_lens)
print(f"avg chars before: {avg_before:.0f}")
print(f"avg chars after:  {avg_after:.0f}")
print(f"avg reduction:    {(1 - avg_after/avg_before):.1%}")
print(f"max after:        {max(after_lens)}")
print(f"no match: {len(no_match_ids)}/{len(paper_text)}")

# Don't trust the aggregate — actually look at why these failed
for pid in no_match_ids[:3]:
    print(f"\n--- {pid} tail (last 400 chars) ---")
    print(paper_text[pid][-400:])

avg chars before: 59714
avg chars after:  38519
avg reduction:    35.5%
max after:        219303
no match: 32/2000

--- gItvr7Xl66 tail (last 400 chars) ---
space by both vr head-mounted display and external users. In Proceedings of the 31st Annual ACM Symposium on User Interface Software and Technology, pp. 499-509, 2018.

[55] H. Zhu, W. Jin, M. Xiao, S. Murali, and M. Li. Blinkey: A two-factor user authentication method for virtual reality devices. Proceedings of the ACM on Interactive, Mobile, Wearable and Ubiquitous Technologies, 4(4):1-29, 2020.

--- Byk-VI9eg tail (last 400 chars) ---
ReLu activations for all the hidden units. Tanh activation at the output units of the generator. Sigmoid at the output of the Discriminator.

- Training was performed with Adam (Kingma &Ba (2014)) (lr $= 2 \times  {10}^{-4},{\beta }_{1} = {0.5}$ ).

- MNIST was trained for 20 epochs with a minibatch of size 100 .

- CelebA and CIFAR were trained over 24000 iterations with a minibatch of size 100 .


In [ ]:
## Creating a separate column
# Convert to a standard list of dicts so in-place updates are kept
sample_list = list(sample)

for r in sample_list:

  pid = r["paper_id"]
  # def strip_refs_and_appendix(paper_text) to do !
  r["paper_text"] = paper_text.get(pid)
  r["match_status"] = "matched" if pid in paper_text else "unmatched"

status_counts = Counter(r["match_status"] for r in sample_list)
print(f"final status on {len(sample_list)} records : {dict(status_counts)}")


final status on 2000 records : {'matched': 2000}


In [ ]:
import os
import json

# Define target directory on Drive
extraction_save_dir = "/content/drive/MyDrive/rc_project_extracted_data_V3"
os.makedirs(extraction_save_dir, exist_ok=True)

# Save the raw extracted paper texts mapping
with open(os.path.join(extraction_save_dir, "extracted_paper_texts.json"), "w", encoding="utf-8") as f:
    json.dump(paper_text, f, indent=2, ensure_ascii=False)

# Save the matched sample list (containing reviews, paper_ids, metadata, etc.)
with open(os.path.join(extraction_save_dir, "extracted_sample_list.json"), "w", encoding="utf-8") as f:
    json.dump(sample_list, f, indent=2, ensure_ascii=False)

print(f"Successfully saved raw extracted data to: {extraction_save_dir}")

Successfully saved raw extracted data to: /content/drive/MyDrive/rc_project_extracted_data_V3


### 1. Installing a Training Stack


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!pip install -q --upgrade "transformers>=4.51.0" "peft>=0.17.0" "accelerate>=1.7.0" "bitsandbytes>=0.46.0" "datasets>=3.6.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 129.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 46.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [ ]:
!pip install -q liger-kernel


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 910.1/910.1 kB 19.2 MB/s eta 0:00:00


In [ ]:
!pip install 'liger-kernel[cutedsl]'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 101.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 59.6 MB/s eta 0:00:00


In [ ]:
import hashlib
import platform
from importlib.metadata import version
from packaging.version import Version

# import bitsandbytes
import numpy as np
import torch
import transformers
from datasets import Dataset, load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,

)

for package in ("transformers", "peft", "accelerate",  "datasets"):
  print(f"{package}: {version(package)}")
print(f"Python Version {platform.python_version()}")
print(f"PyTorch Version {torch.__version__}")
print(f"GPU:{torch.cuda.get_device_name(0)}")
print(f"GPU Mem: {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB")

transformers: 5.20.2
peft: 0.21.2
accelerate: 1.15.0
datasets: 5.1.0
Python Version 3.13.15
PyTorch Version 2.11.0+cu130
GPU:NVIDIA A100-SXM4-40GB
GPU Mem: 39.5 GB


### Defnining Profile


In [ ]:
# Defining Constants
SEED = 42

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
MAX_LENGTH = 18432
MAX_PAPER_TOKENS = 16050
MAX_TARGET_TOKENS = 1750
MAX_NEW_TOKENS = 16384
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4

MAX_TRAIN_ROWS = 2800
MAX_EVAL_ROWS = 280
MAX_TEST_ROWS = 280
np.random.seed(SEED)
torch.manual_seed(SEED)


In [ ]:

# Tokenizer Setup
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Bits and Bytes Configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

from liger_kernel.transformers import apply_liger_kernel_to_qwen3
apply_liger_kernel_to_qwen3()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config = quantization_config,
    device_map = "auto", #: run as a plain single-process script, and don't launch with accelerate launch --multi_gpu or torchrun, because those conflict with device_map="auto"'s own placement logic
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
print(f"Loaded Model Footprint (4B): {model.get_memory_footprint() / 2**30:.2f} GB")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Loaded Model Footprint (4B): 2.42 GB


In [ ]:
print("Model:", MODEL_ID)
print("Tokenizer EOS: ", tokenizer.eos_token, tokenizer.eos_token_id)
print("Tokenizer PAD: ", tokenizer.pad_token, tokenizer.pad_token_id)
print("Generation EOS: ", model.generation_config.eos_token_id)

probe = tokenizer.apply_chat_template(
    [{
        "role": "user", "content": "write a short review."
    }],
    tokenize=False,
    add_generation_prompt=True,
)

print("Prompt ending:", repr(probe[-100:]))
assert "<think>" not in probe[-100: ]

Model: Qwen/Qwen3-4B-Instruct-2507
Tokenizer EOS:  <|im_end|> 151645
Tokenizer PAD:  <|endoftext|> 151643
Generation EOS:  [151645, 151643]
Prompt ending: '<|im_start|>user\nwrite a short review.<|im_end|>\n<|im_start|>assistant\n'


### 2. Load and Split the Review data


In [ ]:
import random

# stripping references and appendix sections to save context cost
paper_text = {
    pid: strip_refs_and_appendix(text)
    for pid, text in paper_text.items()
}

for r in sample_list:
  pid = r["paper_id"]
  r["paper_text"] = paper_text.get(pid)
  r["match_status"] = "matched" if pid in paper_text else "unmatched"

# Loading the dataset and checking if the required columns for the training are present
raw_dataset = Dataset.from_list(sample_list)
raw_dataset = raw_dataset.filter(lambda r: r["match_status"] == "matched")
print(f"{len(raw_dataset)} matched records after filtering")

required_columns = {"paper_id", "paper_text", "reviews"}
assert required_columns.issubset(raw_dataset.column_names), raw_dataset.column_names

assert all(
    isinstance(r["paper_text"], str) and r["paper_text"].strip()
    for r in raw_dataset
), "paper_text is not a non-empty string"


# Creating Groups by paper-> review
rows = []
for example in raw_dataset:
  paper_id = example["paper_id"]
  paper_text_val = example["paper_text"]

  for i, review in enumerate(example["reviews"]):
    rows.append({
        'unit_id': f"{paper_id}_{i}",
        'paper_id': paper_id,  # changed
        'whole_paper': paper_text_val,
        'review_content': review["review_content"],
        'meta': {
            'review_title': review['review_title'],
            'initial_score': review["initial_score"],
            'final_score': review["final_score"],
            'initial_score_unified': review["initial_score_unified"],
            'final_score_unified': review["final_score_unified"],
        }
    })

# Build dataset from the list of rows
dataset = Dataset.from_list(rows)
dataset = dataset.filter(lambda r: len((r["review_content"] or "").strip()) >= 100) #changed
print(f"{len(dataset)}: paper, review rows built")

# Properly shuffle and split the HF Dataset
# shuffled_dataset = dataset.shuffle(seed=SEED)

# total_len = len(shuffled_dataset)
# train_end = min(MAX_TRAIN_ROWS, total_len)
# val_end = min(train_end + MAX_EVAL_ROWS, total_len)
# test_end = min(val_end + MAX_TEST_ROWS, total_len)

# train_dataset = shuffled_dataset.select(range(0, train_end))
# val_dataset = shuffled_dataset.select(range(train_end, val_end))
# test_dataset = shuffled_dataset.select(range(val_end, test_end))

pids = sorted({r["paper_id"] for r in rows}); np.random.default_rng(SEED).shuffle(pids); n = len(pids)
part = {p: ("train" if i< .8*n else "val" if i < .9*n else "test") for i, p in enumerate(pids)}

def take(name, cap):
  data = dataset.filter(lambda r: part[r["paper_id"]] == name)
  data = data.shuffle(seed=SEED)
  return data.select(range(min(cap, len(data))))

train_dataset, val_dataset, test_dataset = take("train", MAX_TRAIN_ROWS), take("val", MAX_EVAL_ROWS), take("test", MAX_TEST_ROWS)
raw_train_dataset = train_dataset
raw_val_dataset = val_dataset
raw_test_dataset = test_dataset

print(f"train: {len(train_dataset)}, val: {len(val_dataset)}, test: {len(test_dataset)}")

# Check for leakage using unit_id sets
train_ids = set(train_dataset['unit_id'])
val_ids = set(val_dataset['unit_id'])
test_ids = set(test_dataset['unit_id'])

pid = lambda ds: {u.rsplit("_", 1)[0] for u in ds["unit_id"]}
print(len(pid(train_dataset) & pid(test_dataset)), "of", len(pid(test_dataset)), "test papers are also in train")

assert len(train_ids & val_ids) == 0, "Leakage found between train and val!"
assert len(train_ids & test_ids) == 0, "Leakage found between train and test!"
assert len(val_ids & test_ids) == 0, "Leakage found between val and test!"
print("No leakage found between splits!")

Filter:   0%|          | 0/2000 [00:00<?, ? examples/s]

2000 matched records after filtering


Filter:   0%|          | 0/7052 [00:00<?, ? examples/s]

7040: paper, review rows built


Filter:   0%|          | 0/7040 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7040 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7040 [00:00<?, ? examples/s]

train: 2800, val: 280, test: 280
0 of 172 test papers are also in train
No leakage found between splits!


### 3. Render Prompts and Mask The Labels


In [ ]:
## old code in comments
# MAX_PAPER_TEXT_CHARS = 25000
# MAX_REVIEW_CHARS = 25000
# cheap cut
# def compact_text(value, max_chars):
#   text = " ".join((value or "").split())
#   if len(text) <= max_chars:
#     return text
#   shortened = text[:max_chars].rsplit(" ", 1)[0].rstrip()
#   return (text[:max_chars] or shortened) + "..."

def clean_text(value):
  lines = (value or "").splitlines()
  # Fix: join the words with space first, then join the lines with newline
  text = "\n".join(" ".join(line.split()) for line in lines).strip()
  while "\n\n\n" in text:
    text = text.replace("\n\n\n", "\n") # Fix: reassign the replaced string
  return text

def trim_to_tokens(text, max_tokens):
  ids = tokenizer(
      text or "",
      add_special_tokens=False,
      truncation=True,
      max_length=max_tokens
  )['input_ids']

  return tokenizer.decode(ids, skip_special_tokens=True)

def prompt_for(example):
  paper = trim_to_tokens(clean_text(example["whole_paper"]), MAX_PAPER_TOKENS)
  user_text = f"""You are an independent academic peer reviewer.

Write a substantive, fair, constructive review of the submitted
manuscript. Ground factual claims in the manuscript. Never invent
results, citations, numerical values, or section, table, or figure
references that are not present.

Prioritize the most consequential issues rather than listing generic
objections. For each major concern, identify the relevant claim,
method, or experiment; explain the evidence gap or reasoning behind
the concern and why it matters; and give an actionable question or
suggestion where appropriate.

Assess the contribution and novelty in the context of related work
actually discussed in the manuscript, methodological validity,
whether experiments support the conclusions, baseline adequacy,
limitations, and reproducibility where assessable.

Distinguish demonstrated problems from questions where the evidence
is insufficient. Avoid boilerplate praise, repeated summaries, and
criticisms that could apply to almost any paper. Use natural
peer-review prose. Use headings or bullets when helpful, but do not
force a fixed template or add a verdict unsupported by the evidence.
Be concise without sacrificing substantive, paper-specific analysis.

Paper:
{paper}"""
  messages = [{'role': 'user', 'content': user_text}]

  return tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True,
  )

def completion_for(example):
  review = trim_to_tokens(clean_text(example["review_content"]), MAX_TARGET_TOKENS,)
  return review + tokenizer.eos_token

def tokenize_with_prompt_mask(example):
  prompt = prompt_for(example)
  full_text = prompt + completion_for(example)
  # with_meta = prompt + example['review_content'] + example['meta']
  full_ids = tokenizer(full_text, add_special_tokens=False)['input_ids']
  paper_ids = tokenizer(clean_text(example["whole_paper"]), add_special_tokens=False)['input_ids']
  review_ids = tokenizer(clean_text(example["review_content"]), add_special_tokens=False)['input_ids']
  paper_was_truncated = len(paper_ids) > MAX_PAPER_TOKENS
  review_was_truncated = len(review_ids) > MAX_TARGET_TOKENS
  encoded = tokenizer(
      full_text,
      add_special_tokens=False,
      truncation=True,
      max_length=MAX_LENGTH,
  )

  prompt_ids = tokenizer(prompt, add_special_tokens=False)['input_ids']
  prompt_length = min(len(prompt_ids), len(encoded['input_ids']))

  assert prompt_ids[:prompt_length] == encoded['input_ids'][:prompt_length], "prompt token boundary mismatch, don't train with misaligned mask"

  labels = list(encoded['input_ids'])
  labels[:prompt_length] = [-100] * prompt_length

  encoded['labels'] = labels
  encoded['untruncated_length'] = len(full_ids)
  encoded['target_tokens'] = sum(token != -100 for token in labels)
  encoded['paper_was_truncated'] = paper_was_truncated
  encoded['review_was_truncated']= review_was_truncated
  return encoded


def make_tokenized(rows):
  tokenized = rows.map(
      tokenize_with_prompt_mask,
      remove_columns=rows.column_names,
      desc="Tokenizing and using Prompt Mask"
  )
  assert all(count > 0 for count in tokenized['target_tokens']), 'a row lost its whole target '
  return tokenized

train_with_stats = make_tokenized(train_dataset)
val_with_stats = make_tokenized(val_dataset)
test_with_stats = make_tokenized(test_dataset)

for name, split in {
    'train': train_with_stats,
    'val': val_with_stats,
    'test': test_with_stats
}.items():
    print(
        f"{name}: "
        f"paper truncation = {sum(split['paper_was_truncated'])}/{len(split)}"
        f"review truncation = {sum(split['review_was_truncated'])}/{len(split)}"
    )
    lengths = np.asarray(split['untruncated_length'])
    truncated = int(np.sum(lengths > MAX_LENGTH))
    print(f"{name}: \n p50 = {np.percentile(lengths, 50):.0f}\n p90 = {np.percentile(lengths, 90):.0f}\n p10 = {np.percentile(lengths, 10):.0f}\n p100 = {np.percentile(lengths, 100):.0f}\n truncated = {truncated}/{len(split)}")

columns_to_drop = ['untruncated_length', 'target_tokens', 'paper_was_truncated', 'review_was_truncated', ]
train_dataset = train_with_stats.remove_columns(columns_to_drop)
val_dataset = val_with_stats.remove_columns(columns_to_drop)
test_dataset = test_with_stats.remove_columns(columns_to_drop)


Tokenizing and using Prompt Mask:   0%|          | 0/2800 [00:00<?, ? examples/s]

Tokenizing and using Prompt Mask:   0%|          | 0/280 [00:00<?, ? examples/s]

Tokenizing and using Prompt Mask:   0%|          | 0/280 [00:00<?, ? examples/s]

train: paper truncation = 198/2800review truncation = 36/2800
train: 
 p50 = 12110
 p90 = 16277
 p10 = 8038
 p100 = 18040
 truncated = 0/2800
val: paper truncation = 17/280review truncation = 6/280
val: 
 p50 = 12453
 p90 = 16352
 p10 = 7300
 p100 = 18040
 truncated = 0/280
test: paper truncation = 27/280review truncation = 2/280
test: 
 p50 = 12131
 p90 = 16526
 p10 = 8569
 p100 = 17261
 truncated = 0/280


In [ ]:
# Re-prepare and verify tokenized datasets in case they were cleared
columns_to_drop = ['untruncated_length', 'target_tokens', 'paper_was_truncated', 'review_was_truncated']
train_dataset = train_with_stats.remove_columns(columns_to_drop)
val_dataset = val_with_stats.remove_columns(columns_to_drop)
test_dataset = test_with_stats.remove_columns(columns_to_drop)

# Inspect the first example
example = train_dataset[0]
visible_text = tokenizer.decode(example['input_ids'], skip_special_tokens=False)
masked_count = sum(label == -100 for label in example['labels'])
target_count = sum(label != -100 for label in example['labels'])
print(visible_text[:1000])
print(f"tokens {len(example['input_ids'])}  prompt-masked {masked_count} review target {target_count}")

# Collator pads -100 values so it never contributes to loss
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
    return_tensors="pt",
    label_pad_token_id=-100,
)
example_batch = data_collator([train_dataset[0], train_dataset[1]])
print(f"shape {[tuple(tensor.shape) for name, tensor in example_batch.items()]}")

<|im_start|>user
You are an independent academic peer reviewer.

Write a substantive, fair, constructive review of the submitted
manuscript. Ground factual claims in the manuscript. Never invent
results, citations, numerical values, or section, table, or figure
references that are not present.

Prioritize the most consequential issues rather than listing generic
objections. For each major concern, identify the relevant claim,
method, or experiment; explain the evidence gap or reasoning behind
the concern and why it matters; and give an actionable question or
suggestion where appropriate.

Assess the contribution and novelty in the context of related work
actually discussed in the manuscript, methodological validity,
whether experiments support the conclusions, baseline adequacy,
limitations, and reproducibility where assessable.

Distinguish demonstrated problems from questions where the evidence
is insufficient. Avoid boilerplate praise, repeated summaries, and
criticisms that could a

In [ ]:
import os
import shutil

# Hygienic configuration for paths
ADAPTER_DIR = '/content/drive/MyDrive/rc_project_adapter_V3'
ZIP_ARCHIVE_PATH = '/content/drive/MyDrive/rc_project_adapter_V3_archive'


### 4. Attaching LoRA Adapters

In [ ]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 40.8 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False,},
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 16,515,072 || all params: 4,038,983,168 || trainable%: 0.4089


### 5. Training with Hugging Face

In [ ]:

# Define a path inside your Drive for the checkpoints
output_dir = "/content/drive/MyDrive/rc_project_LoRA_runs_V3"
model.config.use_cache= False



In [31]:
training_args = TrainingArguments(
    output_dir = output_dir,
    per_device_train_batch_size = TRAIN_BATCH_SIZE,
    per_device_eval_batch_size = EVAL_BATCH_SIZE,
    gradient_accumulation_steps = GRADIENT_ACCUMULATION_STEPS,
    learning_rate = LEARNING_RATE,
    num_train_epochs = 1,
    fp16 = False,
    bf16 = True,
    tf32 = True,
    train_sampling_strategy="batch_rebalance",
    optim = "adamw_torch_fused",
    logging_steps=10,
    logging_first_step = True,
    eval_strategy = "steps",
    eval_steps= 80,
    save_strategy = "steps",
    save_steps = 80,
    load_best_model_at_end=True,
    save_total_limit = 35,
    metric_for_best_model = "eval_loss",
    greater_is_better = False,
    report_to = "none",
    seed=SEED,
    dataloader_num_workers = 2,

)

trainer = Trainer(
    model= model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    data_collator = data_collator,

)

import time, torch


torch.cuda.empty_cache() # just to make some space
torch.cuda.reset_peak_memory_stats()
train_result = trainer.train(transformers.trainer_utils.get_last_checkpoint(output_dir))
# train_result = trainer.train()

print(train_result.metrics)
print("Final step: ", trainer.state.global_step)
print("Peak allocated GiB: ", torch.cuda.max_memory_allocated() / 2**30)
print("Peak reserved GiB", torch.cuda.max_memory_reserved() / 2**30)
print("Validation metrics: ", trainer.evaluate(metric_key_prefix="Validation"))
print("Held out Reviews metrics: ", trainer.evaluate(test_dataset, metric_key_prefix="Test"))

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
320,2.141601,2.166123


Step,Training Loss,Validation Loss
320,2.141601,2.166123
350,2.104433,2.165333


{'train_runtime': 8849.4735, 'train_samples_per_second': 0.099, 'train_steps_per_second': 0.012, 'total_flos': 7.460398826454958e+17, 'train_loss': 2.1122658122669566, 'epoch': 1.0}
Final step:  350
Peak allocated GiB:  29.129165649414062
Peak reserved GiB 30.578125


Training Loss,Validation Loss,Step
2.104433,2.165333,350


Validation metrics:  {'Validation_loss': 2.165332794189453}


Training Loss,Validation Loss,Step
2.104433,2.151797,350


Held out Reviews metrics:  {'Test_loss': 2.15179705619812}


In [32]:
ADAPTER_DIR = '/content/drive/MyDrive/rc_project_adapter_V3'
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("save adapter and tokenizer to: ", ADAPTER_DIR)
import shutil

# Zip the checkpoint folder
shutil.make_archive(
    base_name=ZIP_ARCHIVE_PATH,
    format='zip',
    root_dir=ADAPTER_DIR
)
print(f"Archive created successfully at: {ZIP_ARCHIVE_PATH}.zip")


save adapter and tokenizer to:  /content/drive/MyDrive/rc_project_adapter_V3
Archive created successfully at: /content/drive/MyDrive/rc_project_adapter_V3_archive.zip


In [33]:
from datasets import load_from_disk
import json

source = (
    "/content/drive/MyDrive/rc_project_datasets_V3/"
    "test_dataset_untokenized"
)

output = (
    "/content/drive/MyDrive/rc_project_datasets_V3/"
    "test_dataset_untokenized.jsonl"
)

ds = load_from_disk(source)

with open(output, "w", encoding="utf-8") as f:
    for row in ds:
        f.write(json.dumps({
            "paper_id": str(row["paper_id"]),
            "whole_paper": row["whole_paper"],
            "review_content": row["review_content"],
        }, ensure_ascii=False) + "\n")

print("Exported:", len(ds), "rows")
print("File:", output)

Exported: 280 rows
File: /content/drive/MyDrive/rc_project_datasets_V3/test_dataset_untokenized.jsonl


In [43]:
import torch
import gc

# Delete references to models, trainers, and other GPU tensors
if 'trainer' in globals():
    del trainer
if 'model' in globals():
    del model

gc.collect()
torch.cuda.empty_cache()
print("GPU Memory cleaned successfully!")

GPU Memory cleaned successfully!


### 6. Generating Reviews


In [44]:
from peft import PeftModel

ADAPTER_DIR = '/content/drive/MyDrive/rc_project_adapter_V3'

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, use_fast=True)  # saved alongside the adapter in cell 36
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # generate_reviews_batch flips this to "left" itself

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
)
base_model.config.pad_token_id = tokenizer.pad_token_id

model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.config.use_cache = True

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [45]:
import hashlib
import json
from tqdm.auto import tqdm
import contextlib

BASELINE_N = 25

def seed_for(paper_id):
  return int(hashlib.sha256(str(paper_id).encode()).hexdigest()[:8], 16) % (2**31)
baseline_eval= list(raw_test_dataset.shuffle(seed=SEED).select(range(min(BASELINE_N, len(raw_test_dataset)))))

assert baseline_eval, "No held-out review examples found"

GENERATION_BATCH_SIZE = 5

def generate_reviews(model, examples, max_new_tokens= MAX_NEW_TOKENS, batch_size = GENERATION_BATCH_SIZE, use_adapter = True):
  tokenizer.padding_side = "left"
  model.eval()
  model.config.use_cache = True
  results = []

  adapter_cdx = contextlib.nullcontext() if use_adapter else model.disable_adapter()
  with adapter_cdx:
    for i in tqdm(range(0, len(examples), batch_size), desc="Batches Generation"):
      batch = examples[i:i + batch_size]
      prompts = [prompt_for(ex) for ex in batch]
      inputs = tokenizer(prompts, add_special_tokens=False, return_tensors="pt", padding = True)
      input_device = model.get_input_embeddings().weight.device
      inputs = {name: tensor.to(input_device) for name, tensor in inputs.items()}

      batch_seed = seed_for("_".join(str(ex["paper_id"]) for ex in batch)) # Fixed: example -> ex
      torch.manual_seed(batch_seed)
      if torch.cuda.is_available():
        torch.cuda.manual_seed_all(batch_seed)

      model.eval() # switching model to eval mode to remove training only arguments like dropout
      model.config.use_cache= True # to enable attention cache

      with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample = True,
            temperature = 0.7,
            top_p = 0.8,
            top_k = 20,
            repetition_penalty = 1.1,
            # no_repeat_ngram_size = 4,
            pad_token_id = tokenizer.pad_token_id,
            eos_token_id = model.generation_config.eos_token_id,

        )
        prompt_length = inputs["input_ids"].shape[1]
        for ex, row in zip(batch, generated):
          new_tokens = row[prompt_length:]
          answer = tokenizer.decode(new_tokens, skip_special_tokens=True ).strip()
          results.append({
              "paper_id": str(ex["paper_id"]),
              "answer": answer,
              "input_tokens": prompt_length,
              "output_tokens": int(len(new_tokens)),
              "hit_cap": len(new_tokens) >= max_new_tokens,
          })
  return results

In [46]:
baseline_raw = generate_reviews(model, baseline_eval, use_adapter=False)
finetune_raw = generate_reviews(model, baseline_eval, use_adapter=True)

baseline_results = []
for example, result in zip(baseline_eval, baseline_raw):
  baseline_results.append(
      {
          "paper_id": str(example["paper_id"]),
          "base_review": result["answer"],
          "reference_review": example["review_content"],
          "input_tokens": result["input_tokens"],
          "base_output_tokens": result["output_tokens"],
          "base_hit_cap": result["hit_cap"],
      }
  )

with open ("/content/drive/MyDrive/baseline_results_v3a.json", "w", encoding = "utf-8") as f:
  json.dump(baseline_results, f, indent=2, ensure_ascii=False)
print(f"saved {len(baseline_results)} baseline reviews")

Batches Generation:   0%|          | 0/2 [00:00<?, ?it/s]

Batches Generation:   0%|          | 0/2 [00:00<?, ?it/s]

saved 10 baseline reviews


In [47]:
FINETUNE_N = 10
import json
from tqdm.auto import tqdm

finetune_eval = baseline_eval
finetune_results = []

for example, result in zip(finetune_eval, finetune_raw):
  finetune_results.append(
      {
          "paper_id": str(example["paper_id"]),
          "fine_review": result["answer"],
          "reference_review": example["review_content"],
          "input_tokens": result["input_tokens"],
          "fine_output_tokens": result["output_tokens"],
          "fine_hit_cap": result["hit_cap"],
      }
  )

with open ("/content/drive/MyDrive/finetune_results_v3a.json", "w", encoding = "utf-8") as f:
  json.dump(finetune_results, f, indent=2, ensure_ascii=False)
print(f"saved {len(finetune_results)} finetune evaluation")

saved 10 finetune evaluation


In [48]:
assert len(baseline_results) == len(finetune_results)
assert[(x["paper_id"], x["reference_review"]) for x in baseline_results
] == [ (x["paper_id"], x["reference_review"]) for x in finetune_results], "Baseline and Fine-tuned rows do not match"


comparison = []

for base, fine in zip (baseline_results, finetune_results):
  comparison.append({
      "paper_id": base["paper_id"],
      "base_review": base["base_review"],
      "fine_review": fine["fine_review"],
      "reference_review": base["reference_review"],
      "base_output_tokens": base["base_output_tokens"],
      "fine_output_tokens": fine["fine_output_tokens"],
      "base_hit_cap": base["base_hit_cap"],
      "fine_hit_cap": fine["fine_hit_cap"],
  })

with open("/content/drive/MyDrive/comparison_v3a.json", "w", encoding = "utf-8") as f:
  json.dump(comparison, f, indent=2, ensure_ascii=False)
print(f"saved {len(comparison)} comparison")

saved 10 comparison


In [49]:
import os

# Define the directory on Google Drive to save the datasets and tokenizer
save_dir = "/content/drive/MyDrive/rc_project_datasets_V3"
os.makedirs(save_dir, exist_ok=True)

# 1. Save the untokenized original datasets (splits before tokenization)
raw_train_dataset.save_to_disk(os.path.join(save_dir, "train_dataset_untokenized"))
raw_val_dataset.save_to_disk(os.path.join(save_dir, "val_dataset_untokenized"))
raw_test_dataset.save_to_disk(os.path.join(save_dir, "test_dataset_untokenized"))

# 2. Save the tokenized datasets (with alignment stats and labels)
train_with_stats.save_to_disk(os.path.join(save_dir, "train_dataset_tokenized"))
val_with_stats.save_to_disk(os.path.join(save_dir, "val_dataset_tokenized"))
test_with_stats.save_to_disk(os.path.join(save_dir, "test_dataset_tokenized"))

# 3. Save the tokenizer
tokenizer.save_pretrained(save_dir)

print(f"Successfully saved both tokenized and untokenized datasets along with tokenizer to {save_dir}")

Saving the dataset (0/1 shards):   0%|          | 0/2800 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/280 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/280 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2800 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/280 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/280 [00:00<?, ? examples/s]

Successfully saved both tokenized and untokenized datasets along with tokenizer to /content/drive/MyDrive/rc_project_datasets_V3
